In [2]:
import sys
print(sys.executable)

/usr/local/bin/python3.11


In [3]:
import pyspark
print(pyspark.__version__)

3.5.5


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Assignment44") \
        .getOrCreate()

print(spark.version)

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/14 07:46:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.5


In [5]:
import os
print(os.listdir("data"))

['orders.csv', 'order_items.csv', 'products.csv', 'users.csv']


SOAL 1 Data Loading & Profiling Awal

In [6]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType, IntegerType)
from pyspark.sql.functions import col, when, sum

def profiling_dataframe(df, name):
    null_counts = [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)for c in df.columns]
    df.select(null_counts).show()
    print(f"{name} row count: {df.count()}")
    
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("total_amount", DoubleType(), True)
])

df_orders = spark.read.csv(
    "data/orders.csv",
    header=True,
    schema=orders_schema
)
                 
df_orders.show(5)
df_orders.printSchema()
profiling_dataframe(df_orders,"orders")

+--------+-------+----------+------------+
|order_id|user_id|order_date|total_amount|
+--------+-------+----------+------------+
|    O001|   U001|2023-10-01|   3500000.0|
|    O002|   U001|2023-10-15|    400000.0|
|    O003|   U002|2023-10-02|   8500000.0|
|    O003|   U002|2023-10-02|   8500000.0|
|    O004|   U003|2023-10-03|    150000.0|
+--------+-------+----------+------------+
only showing top 5 rows

root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)



26/10/14 07:46:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+-------+----------+------------+
|order_id|user_id|order_date|total_amount|
+--------+-------+----------+------------+
|       1|      1|         0|           0|
+--------+-------+----------+------------+

orders row count: 18


In [7]:
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True)
])

df_order_items = spark.read.csv(
    "data/order_items.csv",
    header=True,
    schema=order_items_schema
)

df_order_items.show(5)
df_order_items.printSchema()
profiling_dataframe(df_order_items,"order_items")

+--------+----------+--------+---------+
|order_id|product_id|quantity|    price|
+--------+----------+--------+---------+
|    O001|      P001|       1|3500000.0|
|    O001|      P001|       1|3500000.0|
|    O002|      P008|       2| 200000.0|
|    O003|      P002|       1|8500000.0|
|    O004|      P003|       1| 150000.0|
+--------+----------+--------+---------+
only showing top 5 rows

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)

+--------+----------+--------+-----+
|order_id|product_id|quantity|price|
+--------+----------+--------+-----+
|       1|         1|       0|    0|
+--------+----------+--------+-----+

order_items row count: 20


In [8]:
users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", DateType(), True)
])

df_users = spark.read.csv(
    "data/users.csv",
    header=True,
    schema=users_schema
)

df_users.show(5)
df_users.printSchema()
profiling_dataframe(df_users,"users")

+-------+-------------+---------+-----------+
|user_id|         name|     city|signup_date|
+-------+-------------+---------+-----------+
|   U001| Andi Pratama| Jakarta | 2023-01-15|
|   U002| Budi Santoso| SURABAYA| 2023-03-20|
|   U002| Budi Santoso| surabaya| 2023-03-20|
|   U003|   Citra Dewi| Bandung | 2023-06-10|
|   U004|Dimas Anggara|  jakarta| 2023-07-01|
+-------+-------------+---------+-----------+
only showing top 5 rows

root
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)

+-------+----+----+-----------+
|user_id|name|city|signup_date|
+-------+----+----+-----------+
|      1|   0|   0|          0|
+-------+----+----+-----------+

users row count: 14


In [9]:
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True)
])

df_products = spark.read.csv(
    "data/products.csv",
    header=True,
    schema=products_schema
)

df_products.show(5)
df_products.printSchema()
profiling_dataframe(df_products,"products")

+----------+-------------+------------+---------+
|product_id| product_name|    category|    price|
+----------+-------------+------------+---------+
|      P001| Smartphone X| ELEKTRONIK |3500000.0|
|      P001| Smartphone X|  Elektronik|3500000.0|
|      P002|   Laptop Pro|  Elektronik|8500000.0|
|      P003|Kemeja Casual|    Fashion | 150000.0|
|      P004| Celana Jeans|     fashion| 250000.0|
+----------+-------------+------------+---------+
only showing top 5 rows

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|         1|           0|       0|    0|
+----------+------------+--------+-----+

products row count: 10


In [10]:
print("Total Order with 0 value:", 
      df_orders.filter(col("total_amount") <= 0).count())
print("Total Order Items with 0 quantity / price:", 
      df_order_items.filter((col("quantity") <= 0) | (col("price") <= 0)).count())

def primary_key_duplicate(df, key):
    duplicate = (df.groupBy(key).count().filter("count > 1").count())
    print(f"Duplicated Primary Key {key}: {duplicate}")

primary_key_duplicate(df_orders, "order_id")
primary_key_duplicate(df_users, "user_id")
primary_key_duplicate(df_products, "product_id")

Total Order with 0 value: 2
Total Order Items with 0 quantity / price: 2
Duplicated Primary Key order_id: 1
Duplicated Primary Key user_id: 1
Duplicated Primary Key product_id: 1


SOAL 2 — Cleaning & Standardization

In [11]:
#Clean and Standarization the Data

from pyspark.sql.functions import lower, trim, initcap

df_order_clean = df_orders.filter(col("order_id").isNotNull() & col("user_id").isNotNull())
df_order_items_clean = df_order_items.filter(col("order_id").isNotNull() & col("product_id").isNotNull())
df_products_clean = df_products.filter(col("product_id").isNotNull())
df_users_clean = df_users.filter(col("user_id").isNotNull())

df_users_clean = df_users_clean.withColumn("city", initcap(trim("city")))
df_products_clean = df_products_clean.withColumn("category", initcap(trim("category")))

In [12]:
#Deduplication of primary key
def before_clean(df):
    uncleaned_table = df.count()
    print(f"Before Cleaned : {uncleaned_table}")

before_clean(df_order_clean)
before_clean(df_order_items_clean)
before_clean(df_products_clean)
before_clean(df_users_clean)

Before Cleaned : 16
Before Cleaned : 18
Before Cleaned : 9
Before Cleaned : 13


In [13]:
df_order_clean = df_order_clean.dropDuplicates(["order_id"])
df_order_items_clean = df_order_items_clean.dropDuplicates(["order_id" ,"product_id"])
df_products_clean = df_products_clean.dropDuplicates(["product_id"])
df_users_clean = df_users_clean.dropDuplicates(["user_id"])

def after_clean(df):
    cleaned_table = df.count()
    print(f"After Cleaned : {cleaned_table}")

after_clean(df_order_clean)
after_clean(df_order_items_clean)
after_clean(df_products_clean)
after_clean(df_users_clean)

After Cleaned : 15
After Cleaned : 17
After Cleaned : 8
After Cleaned : 12


SOAL 3 — Data Enrichment & Feature Engineering

In [14]:
from pyspark.sql.functions import max, datediff

df_order_items_clean = df_order_items_clean.withColumn(
    "total_item_value", col("quantity") * col("price")
)

df_order_clean = df_order_clean.withColumn(
    "order_value_flag", when(col("total_amount") > 500000, "high_value").otherwise("normal")
)
df_latest_order = df_order_clean.groupBy("user_id").agg(max("order_date").alias("order_date_max_per_user"))

In [16]:
df_order_items_clean.show(5)
df_order_clean.show(5)
df_latest_order.show(5)

+--------+----------+--------+---------+----------------+
|order_id|product_id|quantity|    price|total_item_value|
+--------+----------+--------+---------+----------------+
|    O015|      P007|       2| 450000.0|        900000.0|
|    O011|      P001|       1|3500000.0|       3500000.0|
|    O007|      P003|      -1| 150000.0|       -150000.0|
|    O009|      P007|       1| 450000.0|        450000.0|
|    O008|      P004|       1| -50000.0|        -50000.0|
+--------+----------+--------+---------+----------------+
only showing top 5 rows

+--------+-------+----------+------------+----------------+
|order_id|user_id|order_date|total_amount|order_value_flag|
+--------+-------+----------+------------+----------------+
|    O001|   U001|2023-10-01|   3500000.0|      high_value|
|    O002|   U001|2023-10-15|    400000.0|          normal|
|    O003|   U002|2023-10-02|   8500000.0|      high_value|
|    O004|   U003|2023-10-03|    150000.0|          normal|
|    O005|   U004|2023-10-04|    

In [17]:
df_users_clean = df_users_clean.join(
    df_latest_order,
    on="user_id",
    how="left"
)

In [18]:
df_users_clean = df_users_clean.withColumn("user_tenure_days",
                                           datediff(col("order_date_max_per_user"), col("signup_date")))

In [19]:
df_users_clean = df_users_clean.withColumn("tenure_segment",
                when(col("user_tenure_days") <= 30, "new").
                when(col("user_tenure_days").between(31, 180), "mature").otherwise("loyal"))

In [20]:
df_users_clean.show(5)
df_products_clean.printSchema()
df_order_items_clean.printSchema()
df_order_clean.printSchema()
df_latest_order.printSchema()



+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
|user_id|         name|    city|signup_date|order_date_max_per_user|user_tenure_days|tenure_segment|
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
|   U001| Andi Pratama| Jakarta| 2023-01-15|             2023-10-20|             278|         loyal|
|   U002| Budi Santoso|Surabaya| 2023-03-20|             2023-10-21|             215|         loyal|
|   U003|   Citra Dewi| Bandung| 2023-06-10|             2023-10-03|             115|        mature|
|   U004|Dimas Anggara| Jakarta| 2023-07-01|             2023-10-04|              95|        mature|
|   U005|    Eka Putri|   Medan| 2023-08-15|             2023-10-05|              51|        mature|
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
only showing top 5 rows

root
 |-- product_id: string (nullable = true)
 |-- product_name: 

SOAL 4 — Join & Aggregation (Business Metrics)

In [21]:
df_fact = df_order_clean.join(df_order_items_clean,
                              on="order_id",
                              how="inner")

df_fact = df_fact.join(df_products_clean,
                       on="product_id",
                       how="inner")

df_fact = df_fact.join(df_users_clean.select("user_id", "name", "city"),
                       on="user_id",
                       how="left")

In [22]:
df_fact.printSchema()
df_fact.show(5)
print("Fact row count:", df_fact.count())

root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- order_value_flag: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

+-------+----------+--------+----------+------------+----------------+--------+---------+----------------+-----------------+------------+---------+-------------+--------+
|user_id|product_id|order_id|order_date|total_amount|order_value_flag|quantity|    price|total_item_value|     product_name|    category|    price|         name|    city|
+-------+----------+--------+----------+------------+----------------+--------+

In [23]:
from pyspark.sql.functions import sum, countDistinct

daily_metrics = df_fact.groupBy("order_date", "category").agg(
    sum("total_item_value").alias("daily_gmv_category"),
    countDistinct("order_id").alias("daily_orders_category"),
    sum("quantity").alias("daily_items_sold")
)

In [24]:
daily_metrics = daily_metrics.orderBy(
    col("order_date").asc(),
    col("daily_gmv_category").desc()
)

daily_metrics.show(5)

+----------+------------+------------------+---------------------+----------------+
|order_date|    category|daily_gmv_category|daily_orders_category|daily_items_sold|
+----------+------------+------------------+---------------------+----------------+
|2023-10-01|  Elektronik|         3500000.0|                    1|               1|
|2023-10-02|  Elektronik|         8500000.0|                    1|               1|
|2023-10-03|     Fashion|          150000.0|                    1|               1|
|2023-10-04|Rumah Tangga|          300000.0|                    1|               1|
|2023-10-05|Rumah Tangga|         1200000.0|                    1|               1|
+----------+------------+------------------+---------------------+----------------+
only showing top 5 rows



In [26]:
from pyspark.sql.functions import avg
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

city_metrics = df_fact.groupBy("city").agg(
    sum("total_item_value").alias("total_GMV_per_city"),
    avg("total_amount").alias("city_aov"),
    countDistinct("user_id").alias("unique_customer")
)

city_category_gmv = df_fact.groupBy("city", "category").agg(
    sum("total_item_value").alias("category_gmv")
)

window_city = Window.partitionBy("city").orderBy(desc("category_gmv"))

city_category_ranked = city_category_gmv.withColumn(
    "category_rank", rank().over(window_city)
)

top_3_categories = city_category_ranked.filter(col("category_rank") <= 3).orderBy("city", "category_rank")

In [27]:
city_metrics.show(5)

+--------+------------------+---------+---------------+
|    city|total_GMV_per_city| city_aov|unique_customer|
+--------+------------------+---------+---------------+
|Makassar|          900000.0| 900000.0|              1|
| Jakarta|         4850000.0| 970000.0|              3|
|Semarang|          -50000.0|      0.0|              1|
|Surabaya|         9450000.0|2537500.0|              2|
|   Medan|         1200000.0|1200000.0|              1|
+--------+------------------+---------+---------------+
only showing top 5 rows



In [28]:
top_3_categories.show(truncate=False)

+----------+------------+------------+-------------+
|city      |category    |category_gmv|category_rank|
+----------+------------+------------+-------------+
|Bandung   |Elektronik  |3700000.0   |1            |
|Bandung   |Fashion     |0.0         |2            |
|Jakarta   |Elektronik  |4100000.0   |1            |
|Jakarta   |Fashion     |450000.0    |2            |
|Jakarta   |Rumah Tangga|300000.0    |3            |
|Makassar  |Fashion     |900000.0    |1            |
|Medan     |Rumah Tangga|1200000.0   |1            |
|Semarang  |Fashion     |-50000.0    |1            |
|Surabaya  |Elektronik  |8500000.0   |1            |
|Surabaya  |Fashion     |950000.0    |2            |
|Yogyakarta|Rumah Tangga|600000.0    |1            |
+----------+------------+------------+-------------+



SOAL 5 — Window Analytics & Insight Final

In [29]:
from pyspark.sql.functions import dense_rank, round

In [30]:
df_fact_clean = df_order_clean.select("user_id", "order_id", "total_amount")

df_fact_clean.show(5)


+-------+--------+------------+
|user_id|order_id|total_amount|
+-------+--------+------------+
|   U001|    O001|   3500000.0|
|   U001|    O002|    400000.0|
|   U002|    O003|   8500000.0|
|   U003|    O004|    150000.0|
|   U004|    O005|    300000.0|
+-------+--------+------------+
only showing top 5 rows



In [31]:
user_performance = df_fact_clean.groupBy("user_id").agg(
    sum("total_amount").alias("user_total_spending"),
    countDistinct("order_id").alias("user_total_orders"),
    round(avg("total_amount"),2).alias("user_avg_order_value")
)

window_user = Window.orderBy(desc("user_total_spending"))

user_rank_global = user_performance.withColumn(
    "user_rank", 
    dense_rank().over(window_user)
)

In [32]:
user_rank_global.show(5)

26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------------------+-----------------+--------------------+---------+
|user_id|user_total_spending|user_total_orders|user_avg_order_value|user_rank|
+-------+-------------------+-----------------+--------------------+---------+
|   U002|          8750000.0|                2|           4375000.0|        1|
|   U001|          4100000.0|                3|          1366666.67|        2|
|   U010|          3700000.0|                1|           3700000.0|        3|
|   U005|          1200000.0|                1|           1200000.0|        4|
|   U012|           900000.0|                1|            900000.0|        5|
+-------+-------------------+-----------------+--------------------+---------+
only showing top 5 rows



26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [33]:
gmv_daily = df_fact.groupBy("order_date").agg(
    sum("total_item_value").alias("gmw_daily")
)

window_daily = (Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow))

daily_cumulative_gmv = gmv_daily.withColumn("cumulative_gmv", sum("gmw_daily").over(window_daily)).orderBy("order_date")

In [34]:
daily_cumulative_gmv.show(truncate=False)

26/10/14 07:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 0

+----------+---------+--------------+
|order_date|gmw_daily|cumulative_gmv|
+----------+---------+--------------+
|2023-10-01|3500000.0|3500000.0     |
|2023-10-02|8500000.0|1.2E7         |
|2023-10-03|150000.0 |1.215E7       |
|2023-10-04|300000.0 |1.245E7       |
|2023-10-05|1200000.0|1.365E7       |
|2023-10-06|-150000.0|1.35E7        |
|2023-10-07|-50000.0 |1.345E7       |
|2023-10-08|700000.0 |1.415E7       |
|2023-10-09|450000.0 |1.46E7        |
|2023-10-10|3700000.0|1.83E7        |
|2023-10-15|400000.0 |1.87E7        |
|2023-10-20|200000.0 |1.89E7        |
|2023-10-21|250000.0 |1.915E7       |
|2023-10-22|600000.0 |1.975E7       |
|2023-10-23|900000.0 |2.065E7       |
+----------+---------+--------------+



26/10/14 07:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [35]:
top_10_customers = user_rank_global.join(
    df_users_clean.select("user_id", "name", "city"),
    on="user_id", how="left"
    ).filter(
        col("user_rank") <= 10
    ).select(
        "user_id", "name","city", "user_total_orders","user_total_spending","user_avg_order_value","user_rank").orderBy("user_rank")

top_10_customers.show(truncate=False)

26/10/14 07:51:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 07:51:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/14 0

+-------+-------------+----------+-----------------+-------------------+--------------------+---------+
|user_id|name         |city      |user_total_orders|user_total_spending|user_avg_order_value|user_rank|
+-------+-------------+----------+-----------------+-------------------+--------------------+---------+
|U002   |Budi Santoso |Surabaya  |2                |8750000.0          |4375000.0           |1        |
|U001   |Andi Pratama |Jakarta   |3                |4100000.0          |1366666.67          |2        |
|U010   |Joko Anwar   |Bandung   |1                |3700000.0          |3700000.0           |3        |
|U005   |Eka Putri    |Medan     |1                |1200000.0          |1200000.0           |4        |
|U012   |Lukman Hakim |Makassar  |1                |900000.0           |900000.0            |5        |
|U008   |Hadi Wijaya  |Surabaya  |1                |700000.0           |700000.0            |6        |
|U011   |Kiki Amelia  |Yogyakarta|1                |600000.0    

Dari tabel ini dapat dilihat bahwa kota yang mendominasi adalah Jakarta dan Bandung sebagai kota kedua. AOV bisa terbilang cukup tinggi tetapi user hanya melakukan transaksi 1x - 3x. Dari hal tersebut diketahui bahwa user melakukan repeat order terbilang rendah. Sehingga bisnis dapat berfokus pada customer retention dan mendorong customer untuk melakukan repeat order dengan adanya program loyalty customer, promosi untuk pelanggan baru dan lama, etc